<a href="https://colab.research.google.com/github/Naveen-2510/SAMPLE1/blob/main/fine_tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install required libraries
!pip install transformers datasets torch sentencepiece

# Verify GPU availability
import torch
print(f"GPU available: {torch.cuda.is_available()}")
print(f"GPU name: {torch.cuda.get_device_name(0)}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.4/491.4 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 34.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 37.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 111.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
# Download a sample movie script dataset from a public repository
!wget https://raw.githubusercontent.com/sample-repo/movie-scripts-dataset/main/sample_scripts.txt -O movie_scripts.txt

# Verify the download
!ls -lh movie_scripts.txt
!head -n 5 movie_scripts.txt  # Show first 5 lines to verify content

--2025-04-30 05:36:39--  https://raw.githubusercontent.com/sample-repo/movie-scripts-dataset/main/sample_scripts.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.108.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 404 Not Found
2025-04-30 05:36:39 ERROR 404: Not Found.

-rw-r--r-- 1 root root 0 Apr 30 05:36 movie_scripts.txt


In [ ]:
from datasets import Dataset
import pandas as pd

# Read your movie script file
with open('movie_scripts.txt', 'r') as f:
    script_text = f.read()

# Split the script into scenes or chunks (adjust as needed)
script_chunks = [chunk.strip() for chunk in script_text.split('\n\n') if chunk.strip()]

# Create a pandas DataFrame
df = pd.DataFrame({'text': script_chunks})

# Convert to Hugging Face Dataset
dataset = Dataset.from_pandas(df)

# Now proceed with tokenization
from transformers import GPT2Tokenizer

tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
tokenizer.pad_token = tokenizer.eos_token

def tokenize_function(examples):
    return tokenizer(examples['text'], truncation=True, max_length=512, padding='max_length')

tokenized_datasets = dataset.map(tokenize_function, batched=True, remove_columns=['text'])

# For training, create labels by shifting the inputs
tokenized_datasets = tokenized_datasets.map(
    lambda examples: {'labels': examples['input_ids']},
    batched=True
)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

In [ ]:
!pip install --upgrade transformers

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir='./results',
    run_name='gpt2-finetune',  # Different from output_dir
    num_train_epochs=3,
    per_device_train_batch_size=2,
    fp16=torch.cuda.is_available(),
    report_to="none"  # Disables all logging integrations
)

In [ ]:
# Save the model
model.save_pretrained('./fine_tuned_gpt2_movie')
tokenizer.save_pretrained('./fine_tuned_gpt2_movie')

# Load the fine-tuned model for testing
from transformers import pipeline

movie_script_generator = pipeline(
    'text-generation',
    model='./fine_tuned_gpt2_movie',
    tokenizer='./fine_tuned_gpt2_movie',
    device=0 if torch.cuda.is_available() else -1
)

# Generate some movie script text
prompt = "INT. DARK ALLEY - NIGHT\n"
generated = movie_script_generator(
    prompt,
    max_length=200,
    num_return_sequences=1,
    temperature=0.7,
    do_sample=True
)

print("\nGenerated Movie Script:")
print(generated[0]['generated_text'])

Device set to use cuda:0
Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.



Generated Movie Script:
INT. DARK ALLEY - NIGHT

(TO PILOT)

Hey, I'm a big fan of your work on The Walking Dead. (TO PILOT)

You know, I love how you've done your own series with The Walking Dead. And I'm really excited about the fact that you're back. So, I'm really looking forward to seeing what you've done over the years in the show. (TO PILOT)

I mean, I'm looking forward to seeing all of the things you've done. Yeah, I have a bit of a crush on you and I love you. I just think you're a really great person and I'm really enjoying what you're doing. (TO PILOT)

I think you have a lot of work to do, and I'm really looking forward to seeing you in the show.

(Laughter) I know, I know. I know. I know.


In [ ]:
from transformers import pipeline
import torch

# Configure the generator with more "chaotic" parameters
movie_script_generator = pipeline(
    'text-generation',
    model='./fine_tuned_gpt2_movie',
    tokenizer='./fine_tuned_gpt2_movie',
    device=0 if torch.cuda.is_available() else -1,
    truncation=True
)

# Generate with higher randomness
prompt = "INT. DARK ALLEY - NIGHT\n"
generated = movie_script_generator(
    prompt,
    max_length=200,
    num_return_sequences=1,
    temperature=0.9,       # Higher = more random
    top_k=30,             # Fewer top candidates = weirder choices
    top_p=0.95,           # Broader nucleus sampling
    repetition_penalty=1.0, # Allow some repetition
    do_sample=True
)

print("\nImproved Generated Output:")
print(generated[0]['generated_text'])

Device set to use cuda:0



Improved Generated Output:
INT. DARK ALLEY - NIGHT

HUNTER - LADY'S BEDROOM

HUNTER - TOWER OF THE BEDROOM

HUNTER - WATER FILLING

HUNTER - BONNEVILLE

HUNTER - EAST BANK

HUNTER - EAST BANK (HUNTER, PORTER)

HUNTER - LAND OF MICHIGAN

HUNTER - NEW YORK, USA (HUNTER, BRYAN, ROBERT)

HUNTER - NEW YORK, USA (HUNTER, PEARL)

HUNTER - NARUTO, USA (HUNTER, KAYMUS)

HUNTER - NEW YORK, USA (HUNTER, DUNCAN)

HUNTER - NEW YORK, USA (HUNTER, BRENNE)
